# Finale End-to-End-Evaluation der RAG-Pipeline

Dieses Notebook bewertet den **finalen Output** gegenüber dem Ursprungs-Chunk, aus dem die gespeicherte Frage einmalig erzeugt wurde.

`cos_score = cos(Embedding(finale Antwort), Embedding(Ursprungs-Chunk))`

Beide Evaluationsnotebooks verwenden dasselbe persistente, zufällig aus der gesamten Qdrant-Wissensbasis gezogene Frage–Chunk-Testset. Die Factory-Pipeline bleibt vollständig aktiv; Fallzahl, Timeout, Antwortlänge und Revisionsrunden sind für einen Lauf unter 15 Minuten begrenzt.


## 0. Setup, Factory und Laufzeitbudget


In [1]:
import sys
import json
import time
import datetime
import importlib.util
from pathlib import Path
from IPython.display import display, HTML

NOTEBOOK_START = time.monotonic()

ARBEITSVERZEICHNIS = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        pfad
        for pfad in (ARBEITSVERZEICHNIS, ARBEITSVERZEICHNIS.parent)
        if (pfad / "config.yaml").exists() and (pfad / "app").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Projekt-Root nicht gefunden.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.core.config import load_config
from app.core.factory import build_components
from app.implementations.ollama_client import OllamaClient
from test_verzahnung.evaluation_shared import (
    generiere_antwort,
    kosinus_aehnlichkeit,
    kuerzen,
    lade_oder_erstelle_testset,
    restbudget_ok,
    rufe_retriever_auf,
)

# ============================================================
# GEMEINSAME EVALUATIONS-KONFIGURATION
# ============================================================
TESTSET_SIZE = 4
EVAL_CASES = 3
OUTPUT_FORMAT = "standard"
RANDOM_SEED = 42
MAX_RUNTIME_SECONDS = 14 * 60
LLM_TIMEOUT_SECONDS = 90
ANSWER_MAX_TOKENS = 450
TESTSET_NEU_ERSTELLEN = False

STEP_FILE_PATH = PROJECT_ROOT / "test_verzahnung" / "Input_Daten" / "evaluation_gear.step"
TESTSET_PATH = PROJECT_ROOT / "test_verzahnung" / "rag_evaluation_testset.json"
LOG_DIR = PROJECT_ROOT / "test_verzahnung" / "logs"

config_original = load_config(PROJECT_ROOT / "config.yaml")

# Laufzeitbegrenzte Kopie der Produktivkonfiguration:
# gleiche Implementierungen, aber kürzeres Timeout/Tokenbudget und keine dritte Revisionsrunde.
answer_config = config_original.answer_generator.model_copy(
    update={
        "timeout_s": LLM_TIMEOUT_SECONDS,
        "max_tokens": ANSWER_MAX_TOKENS,
        "max_revisions": 0,
    }
)
config = config_original.model_copy(update={"answer_generator": answer_config})

cad_fallback_grund = None
if config.cad_adapter.implementation == "cad_processor_local":
    if importlib.util.find_spec("OCC") is None:
        cad_fallback_grund = f"pythonocc-core/OCC fehlt im Kernel {sys.executable}."
    elif not STEP_FILE_PATH.exists():
        cad_fallback_grund = f"STEP-Datei fehlt: {STEP_FILE_PATH}"

if cad_fallback_grund:
    config = config.model_copy(
        update={
            "cad_adapter": config.cad_adapter.model_copy(
                update={"implementation": "synthetic_json"}
            )
        }
    )
    print(f"WARNUNG: {cad_fallback_grund}")
    print("Für diesen Lauf wird nur der CAD-Adapter auf synthetic_json umgestellt.")

print("Baue die laufzeitbegrenzte Factory-Pipeline auf ...")
components = build_components(config, base_dir=PROJECT_ROOT)
embedder = components.embedder
retriever = components.retriever
answer_gen = components.answer_generator
cad_adapter = components.cad_adapter
synthetic_cad_adapter = components.synthetic_cad_adapter
store = components.vector_store

frage_client = OllamaClient(
    base_url=config.answer_generator.ollama_url,
    timeout_s=45,
)

print("\nKonfiguration:")
print(f"  Modell:            {config.answer_generator.model_name}")
print(f"  Antwortstrategie:  {config.answer_generator.implementation}")
print(f"  Review aktiv:      {config.answer_generator.enable_review}")
print(f"  Revisionsrunden:   {config.answer_generator.max_revisions}")
print(f"  Testset-Größe:     {TESTSET_SIZE}")
print(f"  Fälle pro Lauf:    {EVAL_CASES}")
print(f"  Laufzeitbudget:    {MAX_RUNTIME_SECONDS / 60:.0f} Minuten")
print(f"  LLM-Timeout:       {LLM_TIMEOUT_SECONDS} Sekunden")
print(f"  Antwort-Tokens:    {ANSWER_MAX_TOKENS}")
print(f"  Collection:        {config.vector_store.collection_name}")
print(f"  Gemeinsames Set:   {TESTSET_PATH}")


WARNUNG: pythonocc-core/OCC fehlt im Kernel /Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/bin/python.
Für diesen Lauf wird nur der CAD-Adapter auf synthetic_json umgestellt.
Baue die laufzeitbegrenzte Factory-Pipeline auf ...


/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(



Konfiguration:
  Modell:            llama3.2:3b
  Antwortstrategie:  multi_agent
  Review aktiv:      True
  Revisionsrunden:   0
  Fälle:             4
  Laufzeitbudget:    14 Minuten
  LLM-Timeout:       60 Sekunden
  Antwort-Tokens:    450
  Collection:        knowledge_base
  Gemeinsames Set:   /Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/test_verzahnung/rag_evaluation_testset.json


## 1. Fester CAD-Kontext


In [2]:
# Ein fester CAD-Kontext für alle Fälle und beide Notebooks.
if config.cad_adapter.implementation == "synthetic_json":
    cad_dateien = synthetic_cad_adapter.list_files()
    if not cad_dateien:
        raise FileNotFoundError("Keine synthetischen CAD-Testdaten vorhanden.")
    cad_metadata = synthetic_cad_adapter.load_file(cad_dateien[0])
    cad_quelle = f"synthetisch: {cad_dateien[0].name}"
else:
    cad_metadata = cad_adapter.extract(file_path=STEP_FILE_PATH)
    cad_quelle = f"echte STEP-Datei: {STEP_FILE_PATH.name}"

print(f"CAD-Kontext: {cad_quelle}")
print(json.dumps(cad_metadata, ensure_ascii=False, indent=2))


CAD-Kontext: synthetisch: gear_01.json
{
  "schema_version": "1.0",
  "source_file": "gear_01.step",
  "gear_type": "spur",
  "confidence": 0.95,
  "basic_geometry": {
    "outer_diameter_mm": 44.0,
    "root_diameter_mm": 35.0,
    "pitch_diameter_mm": 40.0,
    "face_width_mm": 20.0,
    "total_width_mm": 20.0,
    "hub_bore_diameter_mm": 10.0,
    "volume_mm3": 28839.8,
    "surface_area_mm2": 5805.7
  },
  "tooth_profile": {
    "num_teeth": 20,
    "module_mm": 2.0,
    "helix_angle_deg": 0.0,
    "pressure_angle_deg": 20.0,
    "tooth_height_mm": 4.5,
    "addendum_mm": 2.0,
    "dedendum_mm": 2.5,
    "profile_shift_x": 0.0,
    "root_fillet_radius_mm": 0.76,
    "tooth_thickness_mm": 3.142
  },
  "topology": {
    "is_internal_gear": false,
    "symmetry_type": "rotational",
    "cone_angle_deg": null,
    "shaft_angle_deg": null,
    "worm_starts": null,
    "keyway_present": true,
    "has_flanges": false
  },
  "material_context": {
    "material": "16MnCr5",
    "mass_kg": 

## 2. Gemeinsames persistentes Frage–Chunk-Testset


In [3]:
# Das Set wird nur erstellt, wenn die Datei fehlt oder TESTSET_NEU_ERSTELLEN=True ist.
# Bei der Erstellung werden zuerst ALLE texttragenden Qdrant-Punkte gelesen und daraus
# mit random.sample reproduzierbar zufällige Frage-Chunk-Paare gezogen.
testset, testset_erstellt = lade_oder_erstelle_testset(
    testset_path=TESTSET_PATH,
    store=store,
    collection_name=config.vector_store.collection_name,
    frage_client=frage_client,
    model_name=config.answer_generator.model_name,
    n_items=TESTSET_SIZE,
    random_seed=RANDOM_SEED,
    neu_erstellen=TESTSET_NEU_ERSTELLEN,
)

if testset["collection_name"] != config.vector_store.collection_name:
    raise ValueError(
        "Das gespeicherte Testset gehört zu einer anderen Collection. "
        "Setze TESTSET_NEU_ERSTELLEN=True bewusst für genau einen Lauf."
    )
if len(testset["items"]) < EVAL_CASES:
    raise ValueError(
        f"Das Testset enthält nur {len(testset['items'])} Fälle, benötigt werden {EVAL_CASES}."
    )

testfaelle = testset["items"][:EVAL_CASES]
print(f"Testset {'erstellt' if testset_erstellt else 'geladen'}: {TESTSET_PATH}")
print(f"Zufallsbasis: {testset['qdrant_chunks_total']} Chunks aus der gesamten Collection")
for item in testfaelle:
    print(f"  {item['id']}: {item['source_name']} S.{item['page_number']} → {item['question']}")


Testset geladen: /Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/test_verzahnung/rag_evaluation_testset.json
Zufallsbasis: 478 Chunks aus der gesamten Collection
  rag_eval_001: DIN ISO 1328 ISO-Toleranzsystem pt.2.pdf S.19 → Was ist der neue Stufenfaktor für die Zweiflanken-Wälzprüfung und wie wird er im Vergleich zur alten Toleranz abgewogen?
  rag_eval_002: ISO 10825_1995-08-00_ML_2839045.pdf S.26 → Wie wird Electric erosion durch den Durchgang eines kleinen elektrischen Stroms verursacht?
  rag_eval_003: DIN ISO 21771_2014-08-00_DE_2144663.pdf S.81 → Was ist die Formel zur Berechnung der minimal zulässigen Zahndickensehne, die den Unterschied zwischen maximaler und minimaler Sehnenlängen berücksichtigt?
  rag_eval_004: DIN ISO 21771_2014-08-00_DE_2144663.pdf S.20 → Was bedeutet die Nummerierung der Zähne in einer bestimmten Richtung bei Zylinderrädern?


## 3. End-to-End-Durchläufe


In [4]:
ergebnisse = []

for index, fall in enumerate(testfaelle, start=1):
    if not restbudget_ok(NOTEBOOK_START, MAX_RUNTIME_SECONDS, reserve_seconds=190):
        print("Laufzeitbudget erreicht; verbleibende Fälle werden nicht mehr gestartet.")
        break

    eintrag = {
        **fall,
        "treffer_anzahl": 0,
        "answer_text": None,
        "cos_score": None,
        "status": "gestartet",
    }
    try:
        treffer = rufe_retriever_auf(retriever, fall["question"], cad_metadata)
        eintrag["treffer_anzahl"] = len(treffer)
        if not treffer:
            eintrag["status"] = "keine Treffer"
            ergebnisse.append(eintrag)
            continue

        antwort = generiere_antwort(
            answer_gen,
            frage=fall["question"],
            treffer=treffer,
            cad_metadata=cad_metadata,
            output_format=OUTPUT_FORMAT,
        )
        answer_text = str(antwort.get("answer_text") or "").strip()
        if not answer_text:
            eintrag["status"] = "leere Antwort"
            ergebnisse.append(eintrag)
            continue

        eintrag["answer_text"] = answer_text
        eintrag["status"] = "gültig"
    except Exception as exc:
        eintrag["status"] = f"Fehler: {type(exc).__name__}: {exc}"
    ergebnisse.append(eintrag)
    print(f"[{index}/{len(testfaelle)}] {eintrag['status']}", flush=True)

gueltige = [e for e in ergebnisse if e["answer_text"]]
if gueltige:
    antwort_vecs = embedder.embed([e["answer_text"] for e in gueltige]).dense_vectors
    chunk_vecs = embedder.embed([e["chunk_text"] for e in gueltige]).dense_vectors
    for eintrag, antwort_vec, chunk_vec in zip(gueltige, antwort_vecs, chunk_vecs):
        eintrag["cos_score"] = kosinus_aehnlichkeit(antwort_vec, chunk_vec)

laufzeit_seconds = time.monotonic() - NOTEBOOK_START
print(f"Abgeschlossen: {len(ergebnisse)}/{len(testfaelle)} Fälle in {laufzeit_seconds / 60:.2f} Minuten.")


[1/4] gültig


multiagent_generate_failed; Fallback auf Single-Pass
Traceback (most recent call last):
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/http

[2/4] gültig


[3/4] gültig


multiagent_generate_failed; Fallback auf Single-Pass
Traceback (most recent call last):
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/http

[4/4] Fehler: ReadTimeout: timed out


Abgeschlossen: 4/4 Fälle in 7.15 Minuten.


## 4. Auswertung


In [5]:
def zeige_ergebnistabelle(ergebnisse, score_spalten):
    kopf = ["Fall", "Quelle / Seite", "Frage", "Treffer", *score_spalten, "Antwort", "Status"]
    kopf_html = "".join(f"<th>{spalte}</th>" for spalte in kopf)
    zeilen = []
    for eintrag in ergebnisse:
        scores = [
            "—" if eintrag.get(spalte) is None else f"{eintrag[spalte]:.4f}"
            for spalte in score_spalten
        ]
        werte = [
            eintrag["id"],
            f"{eintrag['source_name']} / S.{eintrag['page_number']}",
            kuerzen(eintrag["question"], 110),
            eintrag["treffer_anzahl"],
            *scores,
            kuerzen(eintrag.get("answer_text"), 130),
            eintrag.get("status", ""),
        ]
        zellen = "".join(f"<td>{str(wert)}</td>" for wert in werte)
        zeilen.append(f"<tr>{zellen}</tr>")
    display(HTML(
        f'<div style="max-height:650px;overflow:auto;border:1px solid #999;border-radius:5px;">'
        f'<table style="border-collapse:collapse;width:100%;font-size:.84em;">'
        f'<thead style="position:sticky;top:0;background:#e2e2e2;"><tr>{kopf_html}</tr></thead>'
        f'<tbody>{"".join(zeilen)}</tbody></table></div>'
        '<style>th,td{padding:6px 8px;border-bottom:1px solid #ddd;text-align:left;vertical-align:top}'
        'tbody tr:nth-child(even){background:#f5f5f5}</style>'
    ))


In [6]:
import statistics

zeige_ergebnistabelle(ergebnisse, ["cos_score"])
scores = [e["cos_score"] for e in ergebnisse if e["cos_score"] is not None]
aggregat = {
    "n_valid": len(scores),
    "mean": statistics.mean(scores) if scores else None,
    "median": statistics.median(scores) if scores else None,
    "min": min(scores) if scores else None,
    "max": max(scores) if scores else None,
    "std": statistics.pstdev(scores) if scores else None,
    "anteil_ge_0_50": sum(s >= 0.50 for s in scores) / len(scores) if scores else None,
    "anteil_ge_0_65": sum(s >= 0.65 for s in scores) / len(scores) if scores else None,
}
print("Aggregat:", aggregat)


Fall,Quelle / Seite,Frage,Treffer,cos_score,Antwort,Status
rag_eval_001,DIN ISO 1328 ISO-Toleranzsystem pt.2.pdf / S.19,Was ist der neue Stufenfaktor für die Zweiflanken-Wälzprüfung und wie wird er im Vergleich zur alten Toleranz…,5,0.7281,"Der neue Stufenfaktor für die Zweiflanken-Wälzprüfung ist nicht explizit in der Frage aufgeführt, aber es wird erwähnt, dass ein …",gültig
rag_eval_002,ISO 10825_1995-08-00_ML_2839045.pdf / S.26,Wie wird Electric erosion durch den Durchgang eines kleinen elektrischen Stroms verursacht?,5,0.7075,"Electric erosion wird durch den Durchgang eines kleinen elektrischen Stroms verursacht, der die Temperatur in der Umgebung des ge…",gültig
rag_eval_003,DIN ISO 21771_2014-08-00_DE_2144663.pdf / S.81,"Was ist die Formel zur Berechnung der minimal zulässigen Zahndickensehne, die den Unterschied zwischen maxima…",5,0.7291,Die Formel zur Berechnung der minimal zulässigen Zahndickensehne berücksichtigt den Unterschied zwischen maximaler und minimaler …,gültig
rag_eval_004,DIN ISO 21771_2014-08-00_DE_2144663.pdf / S.20,Was bedeutet die Nummerierung der Zähne in einer bestimmten Richtung bei Zylinderrädern?,5,—,,Fehler: ReadTimeout: timed out


Aggregat: {'n_valid': 3, 'mean': 0.7215800900500675, 'median': 0.7281068082752301, 'min': 0.7075300701825297, 'max': 0.7291033916924426, 'std': 0.00994319158471501, 'anteil_ge_0_50': 1.0, 'anteil_ge_0_65': 1.0}


## 5. JSON-Log


In [7]:
LOG_DIR.mkdir(parents=True, exist_ok=True)
zeitstempel = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_pfad = LOG_DIR / f"{zeitstempel}_final_eval.json"

log_daten = {
    "zeitstempel": zeitstempel,
    "testset_path": str(TESTSET_PATH),
    "testset_created_at": testset["created_at"],
    "testset_bei_diesem_lauf_erstellt": testset_erstellt,
    "collection_name": config.vector_store.collection_name,
    "modell": config.answer_generator.model_name,
    "antwortstrategie": config.answer_generator.implementation,
    "output_format": OUTPUT_FORMAT,
    "cad_quelle": cad_quelle,
    "runtime_seconds": laufzeit_seconds,
    "runtime_budget_seconds": MAX_RUNTIME_SECONDS,
    "n_angefordert": len(testfaelle),
    "n_bearbeitet": len(ergebnisse),
    "aggregat": aggregat,
    "eintraege": ergebnisse,
}
log_pfad.write_text(json.dumps(log_daten, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Log gespeichert: {log_pfad}")


Log gespeichert: /Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/test_verzahnung/logs/20260625_192901_final_eval.json


## Zusammenfassung

| Komponente | Zweck |
|---|---|
| Gemeinsames JSON-Testset | Identische, einmalig erzeugte Frage–Chunk-Paare |
| Factory-Retriever | Führt jede gespeicherte Frage gegen die aktuelle Wissensbasis aus |
| Factory-Antwortgenerator | Erzeugt den finalen Output mit Solver und Reviewer |
| Gemeinsamer Embedder | Vergleicht Antwort und Ursprungs-Chunk im selben Vektorraum |
| `cos_score` | Semantische Nähe des finalen Outputs zum ursprünglichen Quellwissen |
| Laufzeitbudget | Startet nach 14 Minuten keine weiteren Fälle |
